# Compare embedding models (Colab GPU)
1. **Runtime > Change runtime type > GPU** (a free T4 handles every model in the list)
2. Run all cells and upload `stories.jsonl` when asked
3. The last cell downloads `embeddings.zip` with one folder of embeddings per model, plus `results.csv` and `qualitative.md`
4. Unzip it into the repo as `embeddings/`. Only commit the folder(s) of the model(s) you keep.

In [1]:
!pip install -q -U sentence-transformers transformers
!nvidia-smi -L

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 105.2 MB/s eta 0:00:00
GPU 0: Tesla T4 (UUID: GPU-d6f13b62-b354-5f0c-3f2d-92be89d212b4)


In [2]:
import gc
import json
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import files
from sentence_transformers import SentenceTransformer

# prefix: some models need a task prefix. e5 wants "query: " for symmetric similarity.
# Comment models out to run fewer, or add English-only ones if all stories are English
# (e.g. "BAAI/bge-large-en-v1.5", "sentence-transformers/all-mpnet-base-v2").
MODELS = [
    {"name": "BAAI/bge-m3", "prefix": "", "max_seq_length": 1024, "batch_size": 8},
    {"name": "BAAI/bge-m3", "prefix": "", "max_seq_length": 2048, "batch_size": 8},
]

OUTPUT_DIR = Path("embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)
META_FIELDS = ["tag", "subgenre", "motif"]
EVAL_SAMPLE_SIZE = 500
EVAL_K = 10
SEED = 42

## Load stories
Order is kept exactly as in the file, so row *i* of every embedding matrix is story *i*.

In [3]:
files.upload()  # pick stories.jsonl

stories = []
with open("stories.jsonl", "r") as f:
    for line in f:
        stories.append(json.loads(line))


def as_text(value):
    if isinstance(value, list):
        return " ".join(str(v) for v in value)
    return value or ""


story_ids = [s["id"] for s in stories]
texts = [as_text(s["main_text"]) for s in stories]
print(f"Loaded {len(stories)} stories, {len(set(story_ids))} unique IDs")

# Rough length check (words), to judge whether max_seq_length truncates much
words = np.array([len(t.split()) for t in texts])
print("Words per story, percentiles 50/75/90/99:", np.percentile(words, [50, 75, 90, 99]))

Saving stories.jsonl to stories.jsonl
Loaded 49884 stories, 49884 unique IDs
Words per story, percentiles 50/75/90/99: [  72.  153.  335. 1594.]


## Evaluation helpers
There are no ground-truth labels, so we use a **proxy**: how much the metadata (tag, subgenre, motif) of a story overlaps with its semantic neighbours (Jaccard), and how often the language matches. A random baseline is included for context. This is a sanity check, not the final verdict: read `qualitative.md` too.

In [4]:
def meta_set(story):
    out = set()
    for field in META_FIELDS:
        value = story.get(field)
        if isinstance(value, list):
            out.update(f"{field}={v}" for v in value if v)
        elif value:
            out.add(f"{field}={value}")
    return out


def language_of(story):
    value = story.get("language")
    if isinstance(value, list):
        value = value[0] if value else None
    return value


meta_sets = [meta_set(s) for s in stories]
languages = [language_of(s) for s in stories]

rng = np.random.default_rng(SEED)
eval_idx = rng.choice(len(stories), size=min(EVAL_SAMPLE_SIZE, len(stories)), replace=False)
qual_idx = rng.choice(len(stories), size=min(5, len(stories)), replace=False)


def overlap_scores(neighbours):
    jaccard, same_language = [], []
    for row, i in zip(neighbours, eval_idx):
        a = meta_sets[i]
        for j in row:
            b = meta_sets[j]
            union = a | b
            jaccard.append(len(a & b) / len(union) if union else 0.0)
            same_language.append(languages[i] == languages[j])
    return float(np.mean(jaccard)), float(np.mean(same_language))


def proxy_scores(emb, k=EVAL_K):
    sims = emb[eval_idx] @ emb.T
    sims[np.arange(len(eval_idx)), eval_idx] = -1
    neighbours = np.argsort(-sims, axis=1)[:, :k]
    return overlap_scores(neighbours)


def top_neighbours(emb, i, k=5):
    sims = emb @ emb[i]
    sims[i] = -1
    return np.argsort(-sims)[:k]


# Random baseline: what overlap do you get by picking neighbours blindly?
random_neighbours = rng.integers(0, len(stories), size=(len(eval_idx), EVAL_K))
base_jaccard, base_lang = overlap_scores(random_neighbours)
print(f"Random baseline: jaccard={base_jaccard:.4f}, same_language={base_lang:.3f}")

Random baseline: jaccard=0.0232, same_language=0.301


## Embed with every model

In [5]:
def slug(name):
    return name.replace("/", "__")


def embed(cfg):
    model = SentenceTransformer(
        cfg["name"], device="cuda", trust_remote_code=cfg.get("trust_remote_code", False)
    )
    model.max_seq_length = cfg["max_seq_length"]
    model.half()  # fp16: faster and uses less memory

    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    emb = model.encode(
        [cfg["prefix"] + t for t in texts],
        batch_size=cfg["batch_size"],
        show_progress_bar=True,
        normalize_embeddings=True,  # inner product == cosine similarity in FAISS
    )
    seconds = time.time() - start
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return emb.astype("float32"), seconds, peak_gb


results = []
qualitative = {}

for cfg in MODELS:
    name = cfg["name"]
    print(f"\n=== {name} ===")
    try:
        emb, seconds, peak_gb = embed(cfg)
    except Exception as e:  # one failing model should not stop the others
        print(f"FAILED: {e}")
        results.append({"model": name, "error": str(e)[:200]})
        continue

    out_dir = OUTPUT_DIR / slug(name)
    out_dir.mkdir(exist_ok=True)
    np.save(out_dir / "embeddings.npy", emb.astype("float16"))  # half the size on disk
    with open(out_dir / "story_ids.json", "w") as f:
        json.dump(story_ids, f)
    with open(out_dir / "info.json", "w") as f:
        json.dump({**cfg, "n_stories": len(story_ids), "dim": emb.shape[1]}, f)

    jaccard, same_language = proxy_scores(emb)
    results.append({
        "model": name,
        "dim": emb.shape[1],
        "max_seq_length": cfg["max_seq_length"],
        "minutes": round(seconds / 60, 2),
        "peak_gpu_gb": round(peak_gb, 2),
        "meta_jaccard": round(jaccard, 4),
        "same_language": round(same_language, 3),
    })
    qualitative[name] = {int(i): top_neighbours(emb, i).tolist() for i in qual_idx}
    print(results[-1])

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)
results_df


=== BAAI/bge-m3 ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

{'model': 'BAAI/bge-m3', 'dim': 1024, 'max_seq_length': 1024, 'minutes': 9.27, 'peak_gpu_gb': 1.36, 'meta_jaccard': 0.1511, 'same_language': 0.727}

=== BAAI/bge-m3 ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/6236 [00:00<?, ?it/s]

{'model': 'BAAI/bge-m3', 'dim': 1024, 'max_seq_length': 2048, 'minutes': 11.44, 'peak_gpu_gb': 1.58, 'meta_jaccard': 0.151, 'same_language': 0.729}


,model,dim,max_seq_length,minutes,peak_gpu_gb,meta_jaccard,same_language
0,BAAI/bge-m3,1024,1024,9.27,1.36,0.1511,0.727
1,BAAI/bge-m3,1024,2048,11.44,1.58,0.1510,0.729


## Side-by-side examples
Written to `qualitative.md`: for 5 random stories, the top-5 neighbours per model.

In [6]:
lines = []
for i in qual_idx:
    lines.append(f"## Story {story_ids[i]}: {stories[i].get('title')}\n")
    lines.append(f"{texts[i][:300]}...\n")
    for name, per_story in qualitative.items():
        lines.append(f"**{name}**\n")
        for j in per_story[int(i)]:
            lines.append(f"- {stories[j].get('title')} ({story_ids[j]}) {sorted(meta_sets[j])[:4]}")
        lines.append("")

(OUTPUT_DIR / "qualitative.md").write_text("\n".join(lines))
print("\n".join(lines[:40]))

## Story Item_39875: ['']

Men disputeerde gins en weer over dc deugdt, suyverheyt en waerdije van het porceleyn. R. 'Jae, ick houde het meer in waerde als het silver.' R. 'Hou, hou, mijnheer, als het gebroocken is, wat is het dan?' R. 'Altoos noch beter als silver, want als dat in tweën is, soo werdt het grootste stuck voor ...

**BAAI/bge-m3**

- [''] (Item_48483) ['subgenre=mop', 'tag=breken', 'tag=goud', 'tag=groot']
- ['Dispuyt tusschen een Mulder en een Backer.'] (Item_26708) ['subgenre=mop', 'tag=bakker', 'tag=best', 'tag=betalen']
- [''] (Item_40094) ['subgenre=mop', 'tag=gewicht', 'tag=goud', 'tag=licht']
- [''] (Item_40095) ['subgenre=mop', 'tag=diamant', 'tag=discussie', 'tag=juweel']
- [''] (Item_35330) ['subgenre=mop', 'tag=antwoorden', 'tag=deugd', 'tag=domheid']

## Story Item_10168: ['']

Een man gaat bij het Rijk solliciteren voor ambtenaar.
Waarop die man zegt: "Dat kan, maar dan moet je wel gekeurd worden."
Waarop hij zegt: "Nou, dan kan ik wel naar huis gaan, want z

## Zip and download

In [7]:
with zipfile.ZipFile("embeddings.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, filenames in os.walk(OUTPUT_DIR):
        for filename in filenames:
            z.write(os.path.join(root, filename))

files.download("embeddings.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>